# Comparación 3-way: v5 vs v6 (post-pandemia) vs v6_all (post-pandemia + todos)

Hipótesis a contrastar:

1. **Filtrar pre-pandemia mejora la generalización**: si v6 ≥ v5 en split
   forward, los datos pre-2022 no aportan al régimen actual.
2. **El filtro `compras_historicas >= 3` sigue siendo necesario**: si v6_all
   cae > 1pp respecto a v6, vendedores con 1-2 meses de historia saturan
   el target (regla "1 compra → churn" domina).
3. **Volumen vs limpieza**: v6 pierde más de la mitad de la muestra de v5
   (9.841 vs 23.684). Es un trade-off entre tamaño y especificidad temporal.

Modelo: HGB balanceado.
Protocolos: GroupKFold + split temporal forward (últimos 6 meses test).


## 1. Setup


In [1]:
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from google.cloud import bigquery

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, average_precision_score, f1_score,
    precision_score, recall_score, precision_recall_curve, roc_curve,
    confusion_matrix,
)
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 200)
pd.set_option('display.width', 220)
np.random.seed(42)

PROJECT, DATASET = 'glamour-peru-dw', 'glamour_dw'
RANDOM_STATE = 42
N_SPLITS = 5
HORIZON_CHURN = 6   # meses

VARIANTS = ['v5', 'v6', 'v6_all']

bq = bigquery.Client(project=PROJECT)
dfs = {}
for v in VARIANTS:
    table = f'`{PROJECT}.{DATASET}.training_churn_{v}`'
    dfs[v] = bq.query(f'SELECT * FROM {table}').to_dataframe()
    df_ = dfs[v]
    print(f'{v:<8} → {len(df_):>6,} filas · {df_["id_vendedor"].nunique():>5,} vendedoras · '
          f'{df_["mes_obs"].nunique():>3} meses · churn {df_["churn"].mean():.4f} · '
          f'mes_obs [{df_["mes_obs"].min()} → {df_["mes_obs"].max()}]')


v5       → 23,684 filas · 4,211 vendedoras · 106 meses · churn 0.2750 · mes_obs [2017-01-01 → 2025-10-01]


v6       →  9,841 filas · 1,911 vendedoras ·  46 meses · churn 0.2587 · mes_obs [2022-01-01 → 2025-10-01]


v6_all   → 14,362 filas · 4,224 vendedoras ·  46 meses · churn 0.3658 · mes_obs [2022-01-01 → 2025-10-01]


## 2. Pipeline (idéntico a NB 05/07)


In [2]:
EXCLUDE = {
    'id_vendedor', 'mes_obs', 'mes_rank_obs',
    'fecha_ingreso',
    'id_coordinadora', 'ccodubigeo', 'distrito',
    'churn',
}
CATEGORICAL = ['sexo_vendedor', 'tipo_vendedor', 'departamento', 'provincia']

def prepare(df_: pd.DataFrame):
    feature_cols = [c for c in df_.columns if c not in EXCLUDE]
    numeric_cols = [c for c in feature_cols if c not in CATEGORICAL]
    df_ = df_.copy()
    for c in CATEGORICAL:
        df_[c] = df_[c].astype('string').fillna('NA')
    return df_, feature_cols, numeric_cols

def build_preprocessor(numeric_cols, scale=False):
    num_steps = [('impute', SimpleImputer(strategy='median'))]
    if scale: num_steps.append(('scale', StandardScaler()))
    return ColumnTransformer([
        ('num', Pipeline(num_steps), numeric_cols),
        ('cat', Pipeline([
            ('impute', SimpleImputer(strategy='constant', fill_value='NA')),
            ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False)),
        ]), CATEGORICAL),
    ])

def make_hgb(numeric_cols):
    return Pipeline([('prep', build_preprocessor(numeric_cols, False)),
                     ('clf', HistGradientBoostingClassifier(class_weight='balanced',
                                                            max_iter=400, learning_rate=0.05,
                                                            random_state=RANDOM_STATE,
                                                            early_stopping=False))])

def make_logreg(numeric_cols):
    return Pipeline([('prep', build_preprocessor(numeric_cols, True)),
                     ('clf', LogisticRegression(max_iter=2000, class_weight='balanced',
                                                random_state=RANDOM_STATE, n_jobs=-1))])

def make_dummy(numeric_cols):
    return Pipeline([('prep', build_preprocessor(numeric_cols, False)),
                     ('clf', DummyClassifier(strategy='stratified', random_state=RANDOM_STATE))])

def make_heuristic_scorer():
    def scorer(Xte):
        gap = Xte['meses_desde_compra_previa'].fillna(0).astype(float).values
        return 1.0 - np.exp(-gap / float(HORIZON_CHURN))
    return scorer


## 3. GroupKFold sobre cada variante


In [3]:
cv = GroupKFold(n_splits=N_SPLITS)

def evaluate_gkf(df_, name):
    df_, feature_cols, numeric_cols = prepare(df_)
    X = df_[feature_cols].copy()
    y = df_['churn'].astype(int).values
    groups = df_['id_vendedor'].values

    out = {'variant': name, 'n': len(df_), 'n_vend': df_['id_vendedor'].nunique(),
           'churn_rate': float(y.mean()), 'models': {}}

    # heurística (no requiere fit)
    scorer = make_heuristic_scorer()
    proba_h = scorer(X)
    out['models']['Heurística'] = {
        'auc': roc_auc_score(y, proba_h),
        'ap':  average_precision_score(y, proba_h),
    }

    for model_name, factory in [('LogReg', make_logreg), ('HGB', make_hgb)]:
        oof = np.zeros(len(y), dtype=float)
        fold_aucs, fold_aps = [], []
        for tr, te in cv.split(X, y, groups):
            m = factory(numeric_cols); m.fit(X.iloc[tr], y[tr])
            p = m.predict_proba(X.iloc[te])[:, 1]
            oof[te] = p
            fold_aucs.append(roc_auc_score(y[te], p))
            fold_aps.append(average_precision_score(y[te], p))
        out['models'][model_name] = {
            'auc_mean': float(np.mean(fold_aucs)),
            'auc_std':  float(np.std(fold_aucs)),
            'ap_mean':  float(np.mean(fold_aps)),
            'ap_std':   float(np.std(fold_aps)),
            'auc':      float(roc_auc_score(y, oof)),
            'ap':       float(average_precision_score(y, oof)),
        }
        print(f'{name:<8} {model_name:<8} AUC {out["models"][model_name]["auc"]:.4f}  '
              f'AP {out["models"][model_name]["ap"]:.4f}')
    return out

gkf_results = {}
for v in VARIANTS:
    print(f'\n→ {v}')
    gkf_results[v] = evaluate_gkf(dfs[v], v)

print('\nDone.')



→ v5


v5       LogReg   AUC 0.7455  AP 0.4980


v5       HGB      AUC 0.7465  AP 0.5029

→ v6


v6       LogReg   AUC 0.7742  AP 0.5074


v6       HGB      AUC 0.7639  AP 0.5016

→ v6_all


v6_all   LogReg   AUC 0.7971  AP 0.6670


v6_all   HGB      AUC 0.7910  AP 0.6600

Done.


## 4. Comparación GroupKFold lado a lado


In [4]:
rows = []
for v in VARIANTS:
    r = gkf_results[v]
    for m in ['Heurística', 'LogReg', 'HGB']:
        d = r['models'][m]
        rows.append({
            'variant': v,
            'modelo':  m,
            'n_obs':   r['n'],
            'n_vend':  r['n_vend'],
            'churn_rate': r['churn_rate'],
            'AUC':     d['auc'],
            'PR-AUC':  d['ap'],
            'AUC mean ± std': f"{d.get('auc_mean', d['auc']):.4f} ± {d.get('auc_std', 0):.4f}",
        })
gkf_summary = pd.DataFrame(rows)
print(gkf_summary.to_string(index=False))
print()

# pivot: AUC HGB por variante para comparar rápido
pivot = gkf_summary[gkf_summary['modelo'] == 'HGB'].set_index('variant')[['AUC', 'PR-AUC', 'churn_rate', 'n_obs']]
print('HGB por variante:')
print(pivot.round(4))


variant     modelo  n_obs  n_vend  churn_rate      AUC   PR-AUC  AUC mean ± std
     v5 Heurística  23684    4211    0.275038 0.656780 0.414768 0.6568 ± 0.0000
     v5     LogReg  23684    4211    0.275038 0.745518 0.497996 0.7453 ± 0.0053
     v5        HGB  23684    4211    0.275038 0.746526 0.502926 0.7466 ± 0.0038
     v6 Heurística   9841    1911    0.258714 0.693945 0.439265 0.6939 ± 0.0000
     v6     LogReg   9841    1911    0.258714 0.774249 0.507386 0.7739 ± 0.0060
     v6        HGB   9841    1911    0.258714 0.763906 0.501614 0.7639 ± 0.0130
 v6_all Heurística  14362    4224    0.365826 0.477824 0.430422 0.4778 ± 0.0000
 v6_all     LogReg  14362    4224    0.365826 0.797060 0.666985 0.7970 ± 0.0059
 v6_all        HGB  14362    4224    0.365826 0.791031 0.660008 0.7913 ± 0.0038

HGB por variante:
            AUC  PR-AUC  churn_rate  n_obs
variant                                   
v5       0.7465  0.5029      0.2750  23684
v6       0.7639  0.5016      0.2587   9841
v6_all   

## 5. Split temporal forward sobre cada variante


In [5]:
TEST_WINDOW = 6
GAP = HORIZON_CHURN + 1   # 7

def evaluate_forward(df_, name):
    df_, feature_cols, numeric_cols = prepare(df_)

    last_rank = int(df_['mes_rank_obs'].max())
    test_min  = last_rank - TEST_WINDOW + 1
    train_max = test_min - GAP

    df_tr = df_[df_['mes_rank_obs'] <= train_max]
    df_te = df_[df_['mes_rank_obs'].between(test_min, last_rank)]

    if df_tr.empty or df_te.empty:
        return None

    X_tr, y_tr = df_tr[feature_cols], df_tr['churn'].astype(int).values
    X_te, y_te = df_te[feature_cols], df_te['churn'].astype(int).values

    if y_te.sum() == 0 or y_te.sum() == len(y_te):
        return None

    m = make_hgb(numeric_cols); m.fit(X_tr, y_tr)
    proba = m.predict_proba(X_te)[:, 1]

    auc_blk = roc_auc_score(y_te, proba)
    ap_blk  = average_precision_score(y_te, proba)

    # AUC por mes test
    eval_t = df_te[['mes_rank_obs']].copy()
    eval_t['y'] = y_te
    eval_t['p'] = proba
    per_mes = []
    for r, g in eval_t.groupby('mes_rank_obs'):
        yt, pp = g['y'].values, g['p'].values
        if 0 < yt.sum() < len(yt):
            per_mes.append(roc_auc_score(yt, pp))
    auc_std = float(np.std(per_mes)) if per_mes else float('nan')
    auc_mean_per_mes = float(np.mean(per_mes)) if per_mes else float('nan')

    return {
        'variant': name,
        'n_train': len(df_tr), 'n_test': len(df_te),
        'churn_train': float(y_tr.mean()),
        'churn_test':  float(y_te.mean()),
        'overlap_pct': len(set(df_tr['id_vendedor']) & set(df_te['id_vendedor'])) /
                       max(df_te['id_vendedor'].nunique(), 1) * 100,
        'auc_block': auc_blk, 'ap_block': ap_blk,
        'auc_mean_per_mes': auc_mean_per_mes, 'auc_std_per_mes': auc_std,
        'train_max_rank': train_max, 'test_min_rank': test_min, 'last_rank': last_rank,
    }

fwd_results = {}
for v in VARIANTS:
    fwd_results[v] = evaluate_forward(dfs[v], v)
    r = fwd_results[v]
    if r is None:
        print(f'{v}: insuficiente data para split forward.')
        continue
    print(f"{v:<8} train [{r['n_train']:>5,} filas, churn {r['churn_train']:.3f}] "
          f"→ test [{r['n_test']:>5,} filas, churn {r['churn_test']:.3f}] "
          f"AUC bloque {r['auc_block']:.4f}  std/mes {r['auc_std_per_mes']:.4f}  "
          f"overlap {r['overlap_pct']:.1f}%")


v5       train [21,375 filas, churn 0.274] → test [1,061 filas, churn 0.255] AUC bloque 0.7509  std/mes 0.0308  overlap 74.1%


v6       train [7,532 filas, churn 0.252] → test [1,061 filas, churn 0.255] AUC bloque 0.7424  std/mes 0.0368  overlap 71.8%


v6_all   train [11,325 filas, churn 0.368] → test [1,276 filas, churn 0.320] AUC bloque 0.7614  std/mes 0.0292  overlap 67.4%


## 6. Comparación split forward lado a lado


In [6]:
rows = []
for v in VARIANTS:
    r = fwd_results[v]
    if r is None:
        continue
    rows.append({
        'variant':       v,
        'n_train':       r['n_train'],
        'n_test':        r['n_test'],
        'churn_train':   r['churn_train'],
        'churn_test':    r['churn_test'],
        '|drift|':       abs(r['churn_train'] - r['churn_test']),
        'overlap_pct':   r['overlap_pct'],
        'AUC bloque':    r['auc_block'],
        'PR-AUC bloque': r['ap_block'],
        'AUC mean/mes':  r['auc_mean_per_mes'],
        'std AUC/mes':   r['auc_std_per_mes'],
    })
fwd_summary = pd.DataFrame(rows)
print(fwd_summary.round(4).to_string(index=False))


variant  n_train  n_test  churn_train  churn_test  |drift|  overlap_pct  AUC bloque  PR-AUC bloque  AUC mean/mes  std AUC/mes
     v5    21375    1061       0.2744      0.2554   0.0190      74.1007      0.7509         0.4824        0.7533       0.0308
     v6     7532    1061       0.2519      0.2554   0.0036      71.7626      0.7424         0.4585        0.7458       0.0368
 v6_all    11325    1276       0.3679      0.3197   0.0482      67.4003      0.7614         0.5804        0.7659       0.0292


## 7. Veredicto


In [7]:
def fmt(r, key, fmt_str='.4f'):
    return ('—' if r is None else format(r[key], fmt_str))

print('=' * 90)
print('VEREDICTO 3-WAY: v5 vs v6 (post-pandemia) vs v6_all (post-pandemia + todos)')
print('=' * 90)
print()
print(f"{'Métrica':<35} {'v5':>15} {'v6':>15} {'v6_all':>15}")
print('-' * 90)

# AUC GroupKFold (HGB)
v5_a = gkf_results['v5']['models']['HGB']['auc']
v6_a = gkf_results['v6']['models']['HGB']['auc']
v6a_a = gkf_results['v6_all']['models']['HGB']['auc']
print(f"{'AUC GroupKFold (HGB)':<35} {v5_a:>15.4f} {v6_a:>15.4f} {v6a_a:>15.4f}")

# PR-AUC GroupKFold (HGB)
v5_p = gkf_results['v5']['models']['HGB']['ap']
v6_p = gkf_results['v6']['models']['HGB']['ap']
v6a_p = gkf_results['v6_all']['models']['HGB']['ap']
print(f"{'PR-AUC GroupKFold (HGB)':<35} {v5_p:>15.4f} {v6_p:>15.4f} {v6a_p:>15.4f}")

# churn rate
print(f"{'Churn rate (global)':<35} "
      f"{gkf_results['v5']['churn_rate']:>15.4f} "
      f"{gkf_results['v6']['churn_rate']:>15.4f} "
      f"{gkf_results['v6_all']['churn_rate']:>15.4f}")

# n filas / vendedoras
print(f"{'n filas':<35} "
      f"{gkf_results['v5']['n']:>15,} "
      f"{gkf_results['v6']['n']:>15,} "
      f"{gkf_results['v6_all']['n']:>15,}")
print(f"{'n vendedoras':<35} "
      f"{gkf_results['v5']['n_vend']:>15,} "
      f"{gkf_results['v6']['n_vend']:>15,} "
      f"{gkf_results['v6_all']['n_vend']:>15,}")

# split forward
print()
print(f"{'AUC split forward':<35} "
      f"{fmt(fwd_results['v5'], 'auc_block')!s:>15} "
      f"{fmt(fwd_results['v6'], 'auc_block')!s:>15} "
      f"{fmt(fwd_results['v6_all'], 'auc_block')!s:>15}")
print(f"{'Std AUC por mes (test)':<35} "
      f"{fmt(fwd_results['v5'], 'auc_std_per_mes')!s:>15} "
      f"{fmt(fwd_results['v6'], 'auc_std_per_mes')!s:>15} "
      f"{fmt(fwd_results['v6_all'], 'auc_std_per_mes')!s:>15}")
print(f"{'|Δ churn rate train↔test|':<35} "
      f"{fmt(fwd_results['v5'], '|drift|') if False else format(abs(fwd_results['v5']['churn_train']-fwd_results['v5']['churn_test']), '.4f'):>15} "
      f"{format(abs(fwd_results['v6']['churn_train']-fwd_results['v6']['churn_test']), '.4f'):>15} "
      f"{format(abs(fwd_results['v6_all']['churn_train']-fwd_results['v6_all']['churn_test']), '.4f'):>15}")
print()
print('Decisión queda escrita en la celda markdown siguiente.')


VEREDICTO 3-WAY: v5 vs v6 (post-pandemia) vs v6_all (post-pandemia + todos)

Métrica                                          v5              v6          v6_all
------------------------------------------------------------------------------------------
AUC GroupKFold (HGB)                         0.7465          0.7639          0.7910
PR-AUC GroupKFold (HGB)                      0.5029          0.5016          0.6600
Churn rate (global)                          0.2750          0.2587          0.3658
n filas                                      23,684           9,841          14,362
n vendedoras                                  4,211           1,911           4,224

AUC split forward                            0.7509          0.7424          0.7614
Std AUC por mes (test)                       0.0308          0.0368          0.0292
|Δ churn rate train↔test|                    0.0190          0.0036          0.0482

Decisión queda escrita en la celda markdown siguiente.


### Decisión: **mantener v5** como vigente. v6_all descartada (target trampa). v6 viable pero no domina.

#### Resultados consolidados

| Métrica | v5 | v6 | v6_all |
|---|---:|---:|---:|
| **AUC GroupKFold (HGB)** | 0.7465 | **0.7639** | **0.7910** |
| **AUC split forward (HGB)** | **0.7509** | 0.7424 | 0.7614 |
| **PR-AUC / prevalencia (GroupKFold)** | 1.83× | **1.94×** | 1.80× |
| **PR-AUC / prevalencia (forward)** | **1.89×** | 1.80× | 1.81× |
| **AUC heurística (gap previo)** | 0.6568 | **0.6939** | 0.4778 |
| Std AUC por mes (test) | 0.031 | 0.037 | 0.029 |
| Drift train↔test | 1.9pp | **0.36pp** | 4.8pp |
| n filas / vendedoras | 23 684 / 4 211 | 9 841 / 1 911 | 14 362 / 4 224 |

#### Por qué v6_all es trampa

El AUC de 0.7910 en GroupKFold es *engañoso*:

1. **Churn rate sube a 36.6%** (vs 25.9% en v6). +11pp solo por incluir
   vendedoras con poca historia.
2. **La heurística cae a 0.4778** — peor que aleatorio. Significa que
   "más gap = más probabilidad de churn" deja de aplicar al juntar dos
   poblaciones distintas (regulares vs nuevas).
3. **PR-AUC sube a 0.66 pero el lift sobre prevalencia BAJA** (1.80× vs
   1.94× en v6). El modelo aprende a separar "vendedoras nuevas" (target
   trivialmente alto) de "vendedoras con historia" — no a predecir churn
   dentro de cada grupo.
4. **Drift train↔test salta a 4.8pp**: la población de vendedoras nuevas
   cambia más entre periodos.

→ Conclusión: el filtro `compras_historicas >= 3` *sigue siendo
necesario*. Sin él, el target dual contamina el aprendizaje.

#### Por qué v6 no domina a v5

| Hipótesis | Predicción | Resultado |
|---|---|---|
| Filtrar pre-pandemia mejora generalización | v6 ≥ v5 en split forward | ✗ v6 baja 0.85pp |
| Filtro hist≥3 sigue necesario post-pandemia | v6 > v6_all en LIFT | ✓ v6 1.94× > v6_all 1.80× |
| Volumen de v5 compensa el ruido pre-pandemia | v5 ≈ v6 en GroupKFold | parcial (v6 +1.7pp) |

- **v6 mejora cross-section** (+1.7pp en GroupKFold) y reduce drift
  drásticamente (1.9pp → 0.36pp). Régimen más limpio.
- **v5 mejora extrapolación temporal** (+0.85pp en split forward) gracias
  a 2.4× más datos de entrenamiento. El volumen importa.
- **Empate técnico en lift forward** (1.89× vs 1.80×). Diferencia <1pp.
- **v5 tiene 2.4× más muestra**: 23 684 vs 9 841 filas. Más robusto a
  cambios futuros, mejor cobertura de comportamientos atípicos.

#### Recomendación

**Mantener v5 como dataset productivo**. La ganancia de v6 en cross-section
no compensa la pérdida de volumen y la peor extrapolación forward.

#### Sin embargo, hallazgo accionable

La heurística "gap previo" funciona MUCHO mejor en v6 (0.6939) que en v5
(0.6568) y se rompe en v6_all (0.4778). Esto sugiere:

- **El régimen post-pandemia es más predecible por recencia simple**.
- **Las vendedoras pre-pandemia agregan ruido al patrón de recencia** —
  hay vendedoras que volvieron después de gaps largos (probable efecto
  cuarentena/reactivación).
- **Mezclar vendedoras nuevas y con historia rompe la heurística** (v6_all).

→ **Próximo experimento sugerido**: feature de interacción
`compras_historicas × meses_desde_compra_previa`, o un modelo de dos
etapas (clasificar primero "nueva vs establecida", después aplicar
modelo distinto a cada segmento). Esto puede aportar más valor que
seguir iterando sobre el filtro temporal.
